In [1]:
import torch
import torch.nn as nn 
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import pandas as pd 
import numpy as np 

import re

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(
    log_dir="runs/word2vec"
)


from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

[nltk_data] Downloading package punkt to /home/eshaan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/eshaan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/eshaan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


device(type='cuda')

In [ ]:
vocab_size = 5000
feature_size = 50

def encoder(path, vocab_limit) : 

    folder_path = Path(path)
    txt_files = sorted(folder_path.glob("*.txt"))   


    word_tokenised_sentences = []
    all_words = []
    encoded_sentences = []
    
    for file_path in txt_files:
        
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
            text = re.sub(r"\s+", " ", text).strip()
            sentences = sent_tokenize(text.lower())

        stop_words = set(stopwords.words("english"))


        for sentence in sentences : 
            word_tokenised_sentence = []
            for word in word_tokenize(sentence):
                if word not in stop_words and word.isalpha():
                    word_tokenised_sentence.append(word)
                    all_words.append(word)
                    
            word_tokenised_sentences.append(word_tokenised_sentence)

        
    all_words = pd.Series(all_words)
    freq_words = all_words.value_counts().head(vocab_limit -1).index.to_list()
    freq_words = ["<UNK>"] + freq_words

    word_index_dict = {word : idx for idx,word in enumerate(freq_words)}

    for sentence in word_tokenised_sentences :
        encoded_sentence = []
        for word_token in sentence:
            encoded_sentence.append(
                word_index_dict.get(word_token,word_index_dict["<UNK>"])
            )
        encoded_sentences.append(encoded_sentence)

    return word_index_dict, encoded_sentences    

  
word_index_dict, encoded_sentences = encoder(
    r"../datasets/hp_books",
    vocab_size
)

# encoded_sentences

In [3]:
len(encoded_sentences)

82268

In [4]:
class Cbag_Dataset(Dataset):

    def __init__(self, encoded_sentences, window_size = 4):

        self.samples = []

        for sentence in encoded_sentences :
            for i in range(window_size, len(sentence) - window_size) : 
                target_word = torch.tensor(
                    sentence[i]
                    )

                context_words = torch.tensor(
                    sentence[ i-window_size : i ] + sentence[ i+1  :  i+window_size+1 ]
                    ) 

                self.samples.append((context_words, target_word))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        context, target = self.samples[idx]
        return (
            torch.tensor(context, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )


In [5]:
dataset = Cbag_Dataset(encoded_sentences,2)
print(len(dataset))

train_dataloader = DataLoader(
    dataset=dataset,
    batch_size=32,
    pin_memory=True,
    shuffle=True,
)


319923


In [6]:
class simpleW2V(nn.Module):

    def __init__(self, vocab_count, feature_count):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings = vocab_count,
            embedding_dim = feature_count,
            device= device
        )

        self.output = nn.Linear(
            in_features=feature_count,
            out_features=vocab_count,
            device= device
        )


    def forward(self, x):
        # x shape = (batch_size, context_words)
        
        features_x = self.embedding(x)   # shape = (batch_size, context_words, feature_count)
        features_x = features_x.mean(dim = 1)         # shape = (batch_size, feature_count)
        logits  = self.output(features_x)

        
        return logits 

model = simpleW2V(vocab_size, feature_size)
model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [7]:
print("Vocabulary:", len(word_index_dict))
print("Sentences:", len(encoded_sentences))
print("Training samples:", len(dataset))
print("Batches:", len(train_dataloader))

contexts, targets = next(iter(train_dataloader))

print("Contexts:", contexts.shape)
print("Targets:", targets.shape)

Vocabulary: 5000
Sentences: 82268
Training samples: 319923
Batches: 9998
Contexts: torch.Size([32, 4])
Targets: torch.Size([32])


/tmp/ipykernel_8542/2565846015.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(context, dtype=torch.long),
/tmp/ipykernel_8542/2565846015.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(target, dtype=torch.long)


In [8]:
epochs = 100
global_step = 0

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for context_words, target_word in train_dataloader:

        context_words = context_words.to(device)
        target_word = target_word.to(device)

        logits = model(context_words)

        # print("context_words:", context_words.shape)
        # print("logits:", logits.shape)
        # print("target_word:", target_word.shape)

        loss = criterion(logits, target_word)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        writer.add_scalar(
            "Loss/Batch",
            loss.item(),
            global_step
        )
        global_step+=1


    avg_loss = total_loss / len(train_dataloader)


    writer.add_scalar(
        "Loss/Epoch",
        avg_loss,
        epoch
    )


    print(f"EPOCH : {epoch}, avg-loss : {avg_loss}")

/tmp/ipykernel_8542/2565846015.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(context, dtype=torch.long),
/tmp/ipykernel_8542/2565846015.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(target, dtype=torch.long)


EPOCH : 0, avg-loss : 6.679023105398515
EPOCH : 1, avg-loss : 6.10322803806176
EPOCH : 2, avg-loss : 5.87434820748253
EPOCH : 3, avg-loss : 5.7223729493737725
EPOCH : 4, avg-loss : 5.607937988220966
EPOCH : 5, avg-loss : 5.515973298853458
EPOCH : 6, avg-loss : 5.439015504144912
EPOCH : 7, avg-loss : 5.373346793673043
EPOCH : 8, avg-loss : 5.31639480366662
EPOCH : 9, avg-loss : 5.2657783964820615
EPOCH : 10, avg-loss : 5.221730681723846
EPOCH : 11, avg-loss : 5.181554129777562
EPOCH : 12, avg-loss : 5.146231288789725
EPOCH : 13, avg-loss : 5.113770532903731
EPOCH : 14, avg-loss : 5.0854826476436115
EPOCH : 15, avg-loss : 5.058919321849218
EPOCH : 16, avg-loss : 5.034601213932515
EPOCH : 17, avg-loss : 5.012185976466648
EPOCH : 18, avg-loss : 4.992622416480634
EPOCH : 19, avg-loss : 4.974131557530798
EPOCH : 20, avg-loss : 4.956911938623038
EPOCH : 21, avg-loss : 4.941021893448438
EPOCH : 22, avg-loss : 4.926387361346018
EPOCH : 23, avg-loss : 4.912471732179459
EPOCH : 24, avg-loss : 4.9

In [13]:
def get_weight_vector(w_type, idx):
    if w_type == 'in' :
        w_matrix = model.embedding.weight.detach().cpu()
    else:
        w_matrix = model.output.weight.detach().cpu()

    return w_matrix[idx]


def similarity(w_type,word1, word2) :
    word1_idx = word_index_dict[word1]
    word2_idx = word_index_dict[word2]
    w1 = get_weight_vector(w_type ,word1_idx)
    w2 = get_weight_vector(w_type ,word2_idx)

    return F.cosine_similarity(
        w1.unsqueeze(0),
        w2.unsqueeze(0)
    ).item()




def most_similar(word, matrix_type="in", top_k=10):

    if word not in word_index_dict:
        raise ValueError(f"{word} is not in vocabulary")

    if matrix_type == "in":
        W = model.embedding.weight.detach().cpu()

    elif matrix_type == "out":
        W = model.output.weight.detach().cpu()

    else:
        raise ValueError("matrix_type must be 'in' or 'out'")

    target_idx = word_index_dict[word]
    target_vector = W[target_idx]

    similarities = F.cosine_similarity(
        W,
        target_vector.unsqueeze(0),
        dim=1
    )

    values, indices = torch.topk(
        similarities,
        k=top_k + 1
    )

    idx_to_word = {
        idx: word
        for word, idx in word_index_dict.items()
    }

    results = []

    for score, idx in zip(values, indices):

        candidate_word = idx_to_word[idx.item()]

        # skip the word itself
        if candidate_word == word:
            continue

        results.append(
            (candidate_word, score.item())
        )

        if len(results) == top_k:
            break

    return results

In [22]:
word_index_dict

{'<UNK>': 0,
 'harry': 1,
 'said': 2,
 'ron': 3,
 'potter': 4,
 'hermione': 5,
 'page': 6,
 'rowling': 7,
 'dumbledore': 8,
 'back': 9,
 'could': 10,
 'one': 11,
 'like': 12,
 'looked': 13,
 'would': 14,
 'know': 15,
 'around': 16,
 'got': 17,
 'hagrid': 18,
 'professor': 19,
 'well': 20,
 'see': 21,
 'snape': 22,
 'though': 23,
 'think': 24,
 'still': 25,
 'get': 26,
 'time': 27,
 'looking': 28,
 'right': 29,
 'wand': 30,
 'eyes': 31,
 'weasley': 32,
 'face': 33,
 'going': 34,
 'voice': 35,
 'look': 36,
 'go': 37,
 'room': 38,
 'order': 39,
 'come': 40,
 'malfoy': 41,
 'head': 42,
 'door': 43,
 'thought': 44,
 'voldemort': 45,
 'something': 46,
 'saw': 47,
 'phoenix': 48,
 'fire': 49,
 'behind': 50,
 'never': 51,
 'seemed': 52,
 'sirius': 53,
 'hand': 54,
 'way': 55,
 'away': 56,
 'told': 57,
 'asked': 58,
 'half': 59,
 'turned': 60,
 'toward': 61,
 'last': 62,
 'much': 63,
 'us': 64,
 'two': 65,
 'dark': 66,
 'little': 67,
 'long': 68,
 'knew': 69,
 'even': 70,
 'good': 71,
 'want': 

In [23]:
print(similarity("out", "harry", "potter"))

0.6964318752288818


In [38]:
most_similar('owl',"out",20)

[('firebolt', 0.8591326475143433),
 ('percy', 0.8530763387680054),
 ('horntail', 0.8512274026870728),
 ('weasley', 0.8504883050918579),
 ('bird', 0.8461067080497742),
 ('bill', 0.8417714238166809),
 ('anyone', 0.839651346206665),
 ('wandmaker', 0.8387401103973389),
 ('chimney', 0.8379741907119751),
 ('mouse', 0.8347329497337341),
 ('broomstick', 0.8339400291442871),
 ('charlie', 0.8332527875900269),
 ('archway', 0.8326109647750854),
 ('<UNK>', 0.8322396278381348),
 ('moments', 0.8311508297920227),
 ('errol', 0.8302906155586243),
 ('found', 0.8299416303634644),
 ('less', 0.8298031687736511),
 ('sidecar', 0.8296312689781189),
 ('chin', 0.8294649124145508)]

In [41]:
torch.save({
    "model" : model.state_dict(),
    'window_size' : 2,
    'lr' : 0.001,
    'optimizer' : 'adam',
    'vocab_size' : vocab_size,
    'feature_size' : feature_size
    }, "checkpoint1.pth"
)